<a href="https://colab.research.google.com/github/saraxfl/Notebook/blob/main/Collab1_Bloque2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mónica Sarahi Flores Lara

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from google.colab import files


uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name, skiprows=9)

print("Columnas cargadas correctamente:", df.columns.tolist())

# Filtrar la estación MER

df_mer = df[df['id_station'] == 'MER'].copy()

# Convertir la columna fecha para extraer día u hora
df_mer['date_dt'] = pd.to_datetime(df_mer['date'])
df_mer['fecha'] = df_mer['date_dt'].dt.date
df_mer['hour'] = df_mer['date_dt'].dt.hour

# Pivotar tabla de formato largo a ancho
df_pivot = df_mer.pivot_table(
    index=['fecha', 'hour'],
    columns='id_parameter',
    values='valor',
    aggfunc='mean'
).reset_index()

# Limpieza de datos (filtrar nulos en las variables de interés)
cols_req = ['O3', 'NO', 'NO2', 'NOX', 'CO']
df_clean = df_pivot.dropna(subset=cols_req).copy()

# Asegurar tipo numérico
for c in cols_req:
    df_clean[c] = pd.to_numeric(df_clean[c], errors='coerce')

df_clean = df_clean.dropna(subset=cols_req)

print(f"Registros limpios listos para el modelo: {len(df_clean)}")

# Regresión Lineal (O3 ~ NO + NO2 + NOX + CO)

X = df_clean[['NO', 'NO2', 'NOX', 'CO']]
y = df_clean['O3']
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()
print("REGRESIÓN ")
print(model.summary())

# Factor de Inflación de Varianza (VIF)

vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("VIF DE CADA VARIABLE")
print(vif_data)

# Residuos vs predichos y Breusch-Pagan
fitted_vals = model.fittedvalues
residuals = model.resid

plt.figure(figsize=(8, 4))
plt.scatter(fitted_vals, residuals, alpha=0.3, color='royalblue')
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuos vs predichos')
plt.xlabel('Valores Predichos')
plt.ylabel('Residuos')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

bp_test = het_breuschpagan(residuals, X_const)
labels = ['Estadístico LM', 'p-value LM', 'Estadístico F', 'p-value F']
print(" BREUSCH-PAGAN")
for k, v in zip(labels, bp_test):
    print(f"{k}: {v:.6f}")

# Residuos por hora y calculo de Durbin-Watson

plt.figure(figsize=(12, 4))
plt.plot(residuals.values, alpha=0.5, color='teal')
plt.title('Residuos en tiempo')
plt.xlabel('Índice Temporal')
plt.ylabel('Residuos')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

df_clean['residuals'] = residuals
res_by_hour = df_clean.groupby('hour')['residuals'].mean()

plt.figure(figsize=(8, 4))
res_by_hour.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Promedio de residuos por hora')
plt.xlabel('Hora del Día')
plt.ylabel('Residuo Promedio')
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.show()

dw = durbin_watson(residuals)
print("ESTADÍSTICO DURBIN-WATSON")

# QQ-Plot residuos

fig = sm.qqplot(residuals, line='45', fit=True)
plt.title(' QQ-Plot residuos')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()